# Homework 3: Mining Data Stream
We replicated the [TRIÈST](http://www.kdd.org/kdd2016/papers/files/rfp0465-de-stefaniA.pdf) algorithm with resevoir sampling using the [Social circles: Facebook (Dataset)](https://snap.stanford.edu/data/ego-Facebook.html). The dataset have 4039 nodes, 88234 edges and 1612010 triangles.

## Resevoir Sampling 
Our first task is to implement a resevoir sampling. Data Stream in graph flows as edges. Every timestamp, we received an edge denoted as (u,v), where u, v are nodes. 

In [96]:
import pandas as pd
import numpy as np
import random
import itertools

def reservoir_sampling(u_in, v_in, t, M, reservoir, reservoir_flat):
    """
        u_in, v_in
        t: current edges count
        M: Size of the reservoir
        reservoir: Dict[int, Set[int]]，Adjacency list but in a dict
        reservoir_flat: Set[Tuple[int, int]], Flat set of edges for easy checking

        return bool: whether the edge was added to the reservoir
    """
    # Regularize edge representation
    u, v = min(u_in, v_in), max(u_in, v_in)
    
    edge_was_added = False
    need_delete = True

    if t <= M:
        # Phase 1: Reservoir not full, directly add
        edge_was_added = True
        need_delete = False
    elif np.random.rand() < M / t:
        # Phase 2: Reservoir full, replace with probability M/t
        edge_was_added = True
    
    if not edge_was_added:
        return False
    
    if need_delete:
        # Randomly select an existing edge
        reservoir_flat_list = list(reservoir_flat)
        # Sample a tuple from the flat set
        sample_keys = random.choice(reservoir_flat_list)
        u_out, v_out = sample_keys
        # Remove the edge
        reservoir[u_out].remove(v_out)
        if not reservoir[u_out]:
            del reservoir[u_out]
        reservoir[v_out].remove(u_out)
        if not reservoir[v_out]:
            del reservoir[v_out]
        reservoir_flat.remove((u_out, v_out))

    # Add the new edge
    if u not in reservoir:
        reservoir[u] = set()
    reservoir[u].add(v)
    if v not in reservoir:
        reservoir[v] = set()
    reservoir[v].add(u)
    reservoir_flat.add((u, v))  
        
    return edge_was_added


## TRIEST base algorithm

In [97]:
## Helper function
def calculate_xi(t, M):
    if M < 3 or t < 3:
        return 1.0
    p_inverse = (t * (t - 1) * (t - 2)) / (M * (M - 1) * (M - 2))
    return max(1.0, p_inverse)

def get_neighbors_in_reservoir(node, reservoir):
    # reservoir : {u: {v1, v2, ...}}
    return reservoir.get(node, set())

In [98]:
def triest_base(stream, M):
    reservoir = {}
    reservoir_flat = set()
    t = 0 
    global_triangle_count = 0 

    for u_in, v_in in stream:
        t += 1
        u, v = min(u_in, v_in), max(u_in, v_in)
        
        # 1. Reservoir Sampling 
        edge_was_added = reservoir_sampling(u, v, t, M, reservoir, reservoir_flat)
        
        if edge_was_added:
            # 2. Triangle Counting
            u_neighbors = get_neighbors_in_reservoir(u, reservoir)
            v_neighbors = get_neighbors_in_reservoir(v, reservoir)
            common_neighbors = u_neighbors.intersection(v_neighbors)
            
            xi = calculate_xi(t, M)
    
            global_triangle_count += len(common_neighbors) * xi
            
    return global_triangle_count

## Impr Algorithm


In [99]:
## Helper function
def calculate_eta(t, M):
    if M < 2 or t < 2:
        return 1.0
    p_inverse = (t * (t - 1)) / (M * (M - 1))
    return max(1.0, p_inverse)

In [100]:
def triest_impr(stream, M):
    reservoir = {}
    reservoir_flat = set()
    t = 0 
    global_triangle_count = 0 

    for u_in, v_in in stream:
        t += 1
        u, v = min(u_in, v_in), max(u_in, v_in)
        
        # 1. Triangle Counting: Count no matter adding or not.
        
        u_neighbors = get_neighbors_in_reservoir(u, reservoir)
        v_neighbors = get_neighbors_in_reservoir(v, reservoir)
        common_neighbors = u_neighbors.intersection(v_neighbors)
        
        # Calculate eta
        eta = calculate_eta(t, M)
        
        global_triangle_count += len(common_neighbors) * eta
        
        # 2. Reservoir Sampling (Same as BASE)
        reservoir_sampling(u, v, t, M, reservoir, reservoir_flat)
            
    return global_triangle_count

In [101]:
def load_and_shuffle_data(file_path, num_shuffles=3):
    print(f"--- Loading: {file_path} ---")
    
    edges = pd.read_csv(
        file_path, 
        sep='\s+', 
        comment='#', 
        header=None, 
        names=['u', 'v']
    )
    
    # Standardize edges and remove duplicates
    edges['u'], edges['v'] = np.where(edges['u'] < edges['v'], (edges['u'], edges['v']), (edges['v'], edges['u']))
    edges = edges.drop_duplicates()
    
    edge_list = list(edges[['u', 'v']].itertuples(index=False, name=None))
    print(f"Successfully loaded {len(edge_list)} unique edges.")
    
    shuffled_streams = []
    for i in range(num_shuffles):
        stream = list(edge_list)
        random.shuffle(stream)
        shuffled_streams.append(stream)
        
    return shuffled_streams

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
C:\Users\jiashunkang\AppData\Local\Temp\ipykernel_16496\4118086609.py:6: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',


In [102]:
FILE_PATH = 'facebook_combined.txt' 
GROUND_TRUTH = 1612010  # True triangle count for Facebook dataset
M = 2000  # Size of the reservoir
NUM_TESTS = 5 # Number of tests with different orderings

# 1. Prepare data streams
shuffled_streams = load_and_shuffle_data(FILE_PATH, num_shuffles=NUM_TESTS)

# 2. Run algorithms and aggregate results (same as previous summary code)
base_results = []
impr_results = []

for i, stream in enumerate(shuffled_streams):
    print(f"==================== TEST RUN {i+1} ====================")
    
    # --- TRIEST-BASE TEST ---
    base_estimate = triest_base(stream, M)
    base_results.append(base_estimate)
    base_error = abs(base_estimate - GROUND_TRUTH) / GROUND_TRUTH * 100
    print(f"[{i+1}] BASE estimate: {base_estimate:,.0f} | Error: {base_error:.2f}%")
    
    # --- TRIEST-IMPR TEST ---
    impr_estimate = triest_impr(stream, M)
    impr_results.append(impr_estimate)
    impr_error = abs(impr_estimate - GROUND_TRUTH) / GROUND_TRUTH * 100
    print(f"[{i+1}] IMPR estimate: {impr_estimate:,.0f} | Error: {impr_error:.2f}%")

# 3. Aggregate results
base_avg = np.mean(base_results)
base_std = np.std(base_results)
impr_avg = np.mean(impr_results)
impr_std = np.std(impr_results)

print("==================== Final Results Summary ====================")
print(f"True triangle count: {GROUND_TRUTH:,.0f}")

print(f"TRIEST-BASE ({NUM_TESTS} tests):")
print(f"  Average estimate: {base_avg:,.0f}")
print(f"  Average error: {abs(base_avg - GROUND_TRUTH) / GROUND_TRUTH * 100:.2f}%")
print(f"  Standard deviation (stability): {base_std:,.0f}")

print(f"\nTRIEST-IMPR ({NUM_TESTS} tests):")
print(f"  Average estimate: {impr_avg:,.0f}")
print(f"  Average error: {abs(impr_avg - GROUND_TRUTH) / GROUND_TRUTH * 100:.2f}%")
print(f"  Standard deviation (stability): {impr_std:,.0f}")

--- Loading: facebook_combined.txt ---
Successfully loaded 88234 unique edges.
==================== TEST RUN 1 ====================
[1] BASE estimate: 1,677,275 | Error: 4.05%
[1] IMPR estimate: 1,594,846 | Error: 1.06%
==================== TEST RUN 2 ====================
[2] BASE estimate: 1,457,038 | Error: 9.61%
[2] IMPR estimate: 1,652,434 | Error: 2.51%
==================== TEST RUN 3 ====================
[3] BASE estimate: 1,979,585 | Error: 22.80%
[3] IMPR estimate: 1,529,230 | Error: 5.14%
==================== TEST RUN 4 ====================
[4] BASE estimate: 1,336,132 | Error: 17.11%
[4] IMPR estimate: 1,629,192 | Error: 1.07%
==================== TEST RUN 5 ====================
[5] BASE estimate: 2,163,303 | Error: 34.20%
[5] IMPR estimate: 1,587,444 | Error: 1.52%
==================== Final Results Summary ====================
True triangle count: 1,612,010
TRIEST-BASE (5 tests):
  Average estimate: 1,722,667
  Average error: 6.86%
  Standard deviation (stability): 310,546


## Optional Questions

### What were the challenges you faced when implementing the algorithm?

One of the main challenges was implementing the Reservoir Sampling mechanism correctly to ensure uniform randomness.

Initially, to save memory, I tried to sample an edge by first selecting a random node $u$ from the adjacency dictionary, and then selecting a random neighbor $v$. However, this introduced a bias: edges connected to low-degree nodes had a much higher probability of being selected than those connected to high-degree hubs. This bias led to significant errors in the global triangle estimation.I diagnosed this issue by performing a sanity check: I increased the reservoir size to cover the total edges. In this case, the estimation was correct, which confirmed the counting logic was fine but the sampling logic was flawed.

**The Fix:** To solve this, I maintained a separate flat set of all edges currently in the reservoir. This allowed me to select an edge to replace with strictly equal probability ($1/M$), ensuring the estimator remained unbiased.

### Can the algorithm be easily parallelized? If yes, how? If not, why? Explain.

No, the TRIÈST algorithm cannot be easily parallelized in its current form.

The algorithm relies on a global counter $t$ (total edges seen so far) to determine the insertion probability ($M/t$) and the estimation weight ($\xi_t$). Each step strictly depends on the state of the reservoir after the previous step.

Moreover, **Reservoir Sampling** is not easily mergeable. As we have discovered in the first question, if we cannot **randomly** remove an edge in the reservoir, then a bias will be introduced to estimation. In the parallel case, it is even harder to ensure that we are fair enough (random enough) to select a victim from the existing reservoir.


### Does the algorithm work for unbounded graph streams? Explain.

Yes. 

"Unbounded graph streams" implies the data flow may be arbitrarily long. TRIEST specifically deals with this situation because user only need to state the fixed memory space (M edges) and the algorithm handles unbounded streams with reservoir sampling.

### Does the algorithm support edge deletions? If not, what modification would it need? Explain.

Base and Impr do not support edge deletions

These algorithms rely on the invariant that the reservoir contains a uniform sample of all edges seen so far. If a user deletes an edge that happens to be in the reservoir, we create a "gap" (the reservoir size drops below $M$). Since we have discarded previous edges, we cannot "backfill" this gap to restore the sample size. This breaks the probability assumptions used for the estimation weights ($\xi$).

However, in the paper, it also introduced a fully-dynamic algorithm. 

The key modifications are 

- **Allowing the Reservoir to Shrink**: Instead of trying to refill the gap, the algorithm accepts that the reservoir size might drop below $M$.
- **Dynamic Counter Adjustment**: Instead of a simple global counter $t$, the algorithm uses counters ($d_t$) that can decrease. When the sample size shrinks, the algorithm dynamically adjusts the weighting coefficients (making them larger) to compensate for the smaller sample size. This ensures the estimation remains unbiased even when the reservoir is not full.

